# Experiments decision support

Standalone experiments that inform choices made in `01_explore_atoms.ipynb`.
Kept separate because they are expensive to run and their outputs are
already reflected in the primary pipeline's defaults.

Two experiments:

1. **Legal model comparison** : compares three sentence-transformer models
   (general English, English Legal-BERT, multilingual Legal-XLM-R) on
   tag-pair surfacing and downstream LOO accuracy. Decision recorded:
   `all-mpnet-base-v2` (the baseline) is the working default.

2. **Alpha and top-k sweep** : LOO accuracy across the classifier's
   exact-vs-semantic mixing coefficient and retrieval cutoff. Decision
   recorded: `alpha = 0.5, top_k = 3` are the working defaults.

Re-run only when a decision needs revisiting (new corpus, new model
candidates, new classifier scoring rules).


## Setup


In [1]:
import sys
import os
from pathlib import Path


def find_project_root(marker='src/data/atom_table.py'):
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError(f"Could not find project root (no {marker} in any parent)")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 220)

from src.data.atom_table import (
    build_atom_table, compute_tag_weights,
    compute_tag_similarities, evaluate_classifier_loo,
)

df = build_atom_table()
weights_idf = compute_tag_weights(df, method='idf')
print(f'Loaded {len(df)} atoms across {df["article_id"].nunique()} articles')


Loaded 61 atoms across 22 articles


The alpha sweep (Experiment 2) needs a semantic-similarity table using the
current default encoder. Skip this cell if you only intend to re-run
Experiment 1 as that section loads its own models.


In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
sim_pairs = compute_tag_similarities(df, threshold=0.55, model=model)
print(f'{len(sim_pairs)} tag pairs at threshold 0.55')


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

45 tag pairs at threshold 0.55


## 1. Legal model comparison

Which sentence encoder should we use? The primary pipeline uses
`all-mpnet-base-v2` — a general-purpose English model. (Suggested) Experimenting with legal-domain models to see whether domain adaptation
improves tag matching.

Three models to compare at the same threshold, same corpus:
- **all-mpnet-base-v2** : general English (baseline, ~420 MB)
- **legal-bert-base-uncased** (nlpaueb) : Chalkidis et al.'s Legal-BERT (~440 MB, English legal)
- **legal-xlm-roberta-base** (joelniklaus) : multilingual legal, covers Dutch (~1.1 GB)

First run downloads ~2 GB total. Cached afterward.


In [3]:
LEGAL_MODELS = {
    'baseline_mpnet':    'sentence-transformers/all-mpnet-base-v2',
    'legal_bert_en':     'nlpaueb/legal-bert-base-uncased',
    'legal_xlm_roberta': 'joelniklaus/legal-xlm-roberta-base',
}
COMPARISON_THRESHOLD = 0.55


### 1.1 Run each model

Loop through the models, load each, compute tag similarities at the shared threshold. Any model that fails to load (network issue, wrong path) is skipped gracefully.

In [4]:
import gc

sim_pairs_by_model = {}

for label, model_id in LEGAL_MODELS.items():
    print(f'\n=== {label} ({model_id}) ===')
    m = None
    try:
        m = SentenceTransformer(model_id)
        pairs = compute_tag_similarities(df, threshold=COMPARISON_THRESHOLD, model=m)
        sim_pairs_by_model[label] = pairs
        print(f'  {len(pairs)} pairs at threshold {COMPARISON_THRESHOLD}')
    except Exception as e:
        print(f'  Failed: {type(e).__name__}: {e}')
        sim_pairs_by_model[label] = None
    finally:
        if m is not None:
            del m
        gc.collect()



=== baseline_mpnet (sentence-transformers/all-mpnet-base-v2) ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  45 pairs at threshold 0.55

=== legal_bert_en (nlpaueb/legal-bert-base-uncased) ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  7741 pairs at threshold 0.55

=== legal_xlm_roberta (joelniklaus/legal-xlm-roberta-base) ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: joelniklaus/legal-xlm-roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  4954 pairs at threshold 0.55


### 1.2 Summary of pair counts and similarity stats

In [5]:
summary = []
for label, pairs in sim_pairs_by_model.items():
    if pairs is None:
        summary.append({'model': label, 'status': 'failed', 'n_pairs': 0,
                        'avg_similarity': None, 'max_similarity': None})
    else:
        summary.append({
            'model': label,
            'status': 'ok',
            'n_pairs': len(pairs),
            'avg_similarity': round(pairs['similarity'].mean(), 3) if len(pairs) else None,
            'max_similarity': round(pairs['similarity'].max(), 3) if len(pairs) else None,
        })
pd.DataFrame(summary)


,model,status,n_pairs,avg_similarity,max_similarity
0,baseline_mpnet,ok,45,0.642,0.895
1,legal_bert_en,ok,7741,0.733,0.987
2,legal_xlm_roberta,ok,4954,0.723,0.994


### 1.3 Top pairs per model  qualitative view


In [6]:
for label, pairs in sim_pairs_by_model.items():
    if pairs is None or len(pairs) == 0:
        print(f'\n=== {label}: no pairs ===')
        continue
    print(f'\n=== {label} — top 10 pairs ===')
    print(pairs.head(10).to_string(index=False))



=== baseline_mpnet — top 10 pairs ===
              field                           tag_a                        tag_b  similarity
             actors                        creditor                       debtor      0.8954
explicit_references                     paragraph 1             second paragraph      0.8022
           residual        active medical emergency            medical emergency      0.7837
           residual civilian life-support telemetry      physiological telemetry      0.7407
           temporal                   after request                 upon request      0.7242
           residual                       mandatory                    necessary      0.7215
           temporal                for twenty years     longer than twenty years      0.7192
             actors                         patient   someone other than patient      0.7153
             actors                   care provider designated primary caregiver      0.7089
           residual          ph

### 1.4 Pairs unique to each model


In [7]:
def pair_key(row):
    a, b = sorted([row['tag_a'], row['tag_b']])
    return (row['field'], a, b)

model_pairs_sets = {}
for label, pairs in sim_pairs_by_model.items():
    if pairs is None:
        continue
    model_pairs_sets[label] = {pair_key(r) for _, r in pairs.iterrows()}

for label, this_set in model_pairs_sets.items():
    others = set().union(*(v for k, v in model_pairs_sets.items() if k != label))
    only = this_set - others
    print(f'\n=== Only in {label} ({len(only)}) ===')
    pdf = sim_pairs_by_model[label]
    for field, a, b in list(only)[:8]:
        mask = ((pdf['field'] == field) &
                (((pdf['tag_a'] == a) & (pdf['tag_b'] == b)) |
                 ((pdf['tag_a'] == b) & (pdf['tag_b'] == a))))
        sim = pdf[mask]['similarity'].iloc[0] if mask.any() else None
        print(f'  {field}: {a!r} ~ {b!r}  (sim={sim:.3f})')



=== Only in baseline_mpnet (1) ===
  residual: 'definitively categorized' ~ 'legally classified'  (sim=0.592)

=== Only in legal_bert_en (3121) ===
  residual: 'interventions' ~ 'zero-risk planetary containment'  (sim=0.627)
  residual: 'biological quarantine sequence' ~ 'embryonic genetic profile'  (sim=0.764)
  residual: 'generally accepted views' ~ 'severe neuro-synthetic implant rejection'  (sim=0.590)
  residual: 'cryogenic stasis delirium' ~ 'medical emergency'  (sim=0.732)
  actors: 'orbital stem-cell research division' ~ 'port authority'  (sim=0.735)
  residual: 'cross-referenced' ~ 'unclassified extraterrestrial biological agent'  (sim=0.704)
  residual: 'civilian life-support telemetry' ~ 'request'  (sim=0.592)
  residual: 'application' ~ 'molecular decontamination'  (sim=0.791)

=== Only in legal_xlm_roberta (337) ===
  residual: 'duty' ~ 'material'  (sim=0.958)
  residual: 'active medical emergency' ~ 'explicitly'  (sim=0.553)
  residual: 'material' ~ 'retention'  (sim=0.9

### 1.5 Downstream impact : Leave One Out (LOO) accuracy per model

The bigger question: does switching model improve *classification accuracy*, not just surface more pairs? Re-run leave-one-out evaluation with each model's `sim_pairs` and compare top-1, top-3, MRR.


In [8]:
downstream = []
for label, pairs in sim_pairs_by_model.items():
    if pairs is None:
        downstream.append({'model': label, 'status': 'skipped'})
        continue
    _, s = evaluate_classifier_loo(df, weights=weights_idf, sim_pairs=pairs, alpha=0.5, top_k=3)
    downstream.append({'model': label, 'n_sim_pairs': len(pairs), **s})

pd.DataFrame(downstream)


,model,n_sim_pairs,n_atoms,top1_accuracy,top3_accuracy,mean_reciprocal_rank
0,baseline_mpnet,45,61,0.2787,0.4262,0.3497
1,legal_bert_en,7741,61,0.1803,0.4426,0.3033
2,legal_xlm_roberta,4954,61,0.1967,0.4098,0.2951


### 1.6 Reading the results

Decision rule:

- More pairs AND higher top-1 accuracy → straight win, adopt as default
- More pairs but worse accuracy → the extra pairs are noise, stick with baseline
- Fewer pairs but same accuracy → stricter model, stylistic choice only

Legal-BERT out of the box tends to produce a degenerate embedding space
(most tags look similar to most others). This shows up as very high pair
counts with no downstream improvement.

**Decision on record**: `all-mpnet-base-v2` remains the default. Legal-BERT
degenerates; Legal-XLM-R adds mild coverage without accuracy gains.


## 2. Alpha and top-k sweep

Grid over `alpha` (mixing exact vs semantic) and `top_k` (retrieval cutoff).
`alpha=0` uses semantic only; `alpha=1` uses exact only.

**Decision on record**: `alpha=0.5, top_k=3` are the primary-pipeline defaults.
Accuracy plateaus from alpha≥0.25; the exact channel does most of the useful work.


In [9]:
sweep_rows = []
for a in [0.0, 0.25, 0.5, 0.75, 1.0]:
    for k in [3, 5]:
        _, s = evaluate_classifier_loo(df, weights=weights_idf, sim_pairs=sim_pairs, alpha=a, top_k=k)
        sweep_rows.append({'alpha': a, 'top_k': k, **s})
pd.DataFrame(sweep_rows)


,alpha,top_k,n_atoms,top1_accuracy,top3_accuracy,mean_reciprocal_rank,top5_accuracy
0,0.00,3,61,0.1311,0.2131,0.1667,NaN
1,0.00,5,61,0.1311,NaN,0.1839,0.2951
2,0.25,3,61,0.2623,0.4426,0.3470,NaN
3,0.25,5,61,0.2623,NaN,0.3536,0.4754
4,0.50,3,61,0.2787,0.4262,0.3497,NaN
5,0.50,5,61,0.2787,NaN,0.3604,0.4754
6,0.75,3,61,0.2787,0.4262,0.3497,NaN
7,0.75,5,61,0.2787,NaN,0.3596,0.4754
8,1.00,3,61,0.2623,0.4098,0.3333,NaN
9,1.00,5,61,0.2623,NaN,0.3399,0.4426
